# 🧬 BioLabCalc: 5-Minute Interactive Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kabe89/BioLabCalc/blob/main/examples/quickstart_cookbook.ipynb)
[![PyPI Version](https://img.shields.io/pypi/v/biolabcalc.svg)](https://pypi.org/project/biolabcalc/)
[![GitHub License](https://img.shields.io/github/license/kabe89/BioLabCalc)](https://github.com/kabe89/BioLabCalc/blob/main/LICENSE)

**BioLabCalc** is an enterprise-grade, privacy-preserving computational biology suite, command-line benchtop assistant, interactive Excel generator, and browser-based Electronic Lab Notebook (ELN) / LIMS for daily wet-lab molecular biology, biochemistry, cloning, electrophoresis, and cell culture workflows.

### 🌟 Core Design Principles
- **100% Free & Open-Source (MIT)**: Zero subscription fees, zero vendor lock-in.
- **100% Offline & Air-Gapped**: Runs entirely client-side on benchtop computers without hosting an external server or sending unreleased patent sequences to the cloud.
- **Zero Server Setup**: Works immediately out of the box with zero external database configuration.

## 🚀 1. Installation
Install the latest release of `biolabcalc` from PyPI:

In [ ]:
!pip install -q biolabcalc

## ⚖️ 2. Molecular Weight & Sequence Biophysics
Calculate exact isotopic and standard molecular weights for dsDNA, ssRNA, oligos, and proteins.

In [ ]:
import biolabcalc as blc

dna_seq = "ATGCGATCGATCGATCGATCGATCGATC"
mw_dsdna = blc.dna_mw(dna_seq, double_stranded=True)
gc_pct = blc.gc(dna_seq)
rev_comp = blc.rc(dna_seq)

print(f"Sequence:         {dna_seq} ({len(dna_seq)} bp)")
print(f"Rev-Complement:   {rev_comp}")
print(f"GC Content:       {gc_pct:.1f}%")
print(f"dsDNA MW:         {mw_dsdna.average_mw:,.2f} g/mol (Da)")

# Translate ORF to protein
protein_seq = blc.translate(dna_seq)
mw_protein = blc.protein_mw(protein_seq)
print(f"Translated Peptide: {protein_seq}")
print(f"Protein MW:         {mw_protein.molecular_weight_da:,.2f} Da")

## 🌡️ 3. PCR Kinetics & Annealing Thermodynamics
Formulate master mixes, calculate SantaLucia nearest-neighbor thermodynamic melting temperatures ($T_m$), and determine amplification yields.

In [ ]:
from biolabcalc.primers import analyze_primer

fwd_primer = "ATGCGATCGATCGATC"
primer_stats = analyze_primer(fwd_primer)
print(f"Primer: {fwd_primer}")
print(f"  Length: {primer_stats.length_nt} nt | GC: {primer_stats.gc_percent:.1f}%")
print(f"  Tm (SantaLucia): {primer_stats.tm_celsius:.1f} °C")

# PCR kinetics across 30 cycles
pcr_kinetics = blc.pcr(amplicon_bp=850, template_ng=5.0, cycles=30)
print(f"\nPCR Amplification Yield: {pcr_kinetics.amplicon_yield_ng:.1f} ng ({pcr_kinetics.amplicon_yield_pmol:.2f} pmol)")
print(f"Efficiency: {pcr_kinetics.efficiency_percent:.1f}%")

## ✂️ 4. In-Silico Restriction Digestion & Virtual Gel Migration
Perform multi-enzyme digests on circular plasmids or linear fragments, with virtual electrophoresis lane simulation against standardized molecular weight ladders (1 kb, 100 bp, etc.).

In [ ]:
from biolabcalc.gels import simulate_gel

# Create synthetic plasmid sequence with EcoRI (GAATTC) and BamHI (GGATCC) cut sites
plasmid_seq = "GAATTC" + ("A" * 1400) + "GGATCC" + ("T" * 2600)
digest_res = blc.digest_map(plasmid_seq, enzymes=["EcoRI", "BamHI"])

print(f"Enzymes: EcoRI + BamHI | Total Fragments: {digest_res.fragment_count}")
for idx, frag in enumerate(digest_res.fragments, 1):
    print(f"  Fragment #{idx}: {frag.length_bp} bp ({frag.start_pos}..{frag.end_pos})")

# Simulate virtual agarose gel migration against 1 kb DNA ladder
frag_sizes = [f.length_bp for f in digest_res.fragments]
gel_sim = simulate_gel(sample_sizes=frag_sizes, ladder_key="1kb_dna")
print("\nVirtual Gel Electrophoresis Migration (ASCII Preview):")
print(gel_sim.ascii_visualization)

## 🗺️ 5. Interactive SVG Plasmid Map Rendering
Generate presentation-quality, high-contrast SVG circular plasmid maps with annotated feature tracks, origins, and resistance markers.

In [ ]:
from IPython.display import SVG, display

# Render standard catalog vector (e.g. pUC19, pET-28a(+), pBR322)
svg_code = blc.plasmid_svg("pUC19")

# Display interactive SVG inline in notebook cell
display(SVG(svg_code))

## 📊 6. Protein Quantification (Bradford & BCA Standard Curves)
Fit linear regression standard curves with automatic $R^2$ calculation and unknown sample yield predictions.

In [ ]:
from biolabcalc.protein import fit_standard_curve

# Standards: BSA in ug/mL vs measured OD595
standards = [(0.0, 0.045), (125.0, 0.142), (250.0, 0.238), (500.0, 0.420), (1000.0, 0.795)]
curve = fit_standard_curve(standards, assay_type="Bradford")

print(f"Standard Curve Type: {curve.assay_type}")
print(f"Slope (m):          {curve.slope:.6f}")
print(f"Intercept (b):      {curve.intercept:.4f}")
print(f"Correlation R²:     {curve.r_squared:.4f}")

# Predict concentration of an unknown sample at OD595 = 0.500
prediction = curve.predict(0.500)
print(f"\nSample Unknown (OD595 = 0.500) -> {prediction.concentration_mg_ml:.3f} mg/mL ({prediction.concentration_ug_ml:.1f} µg/mL)")

## 🌐 7. Launching the Local Browser-Based ELN & LIMS
To launch the browser-based Electronic Lab Notebook on your workstation or instrument PC without needing to host an external server:

```bash
# From your command line terminal:
biolabcalc eln
```

Or try the live static demo directly in your browser: [BioLabCalc Live Demo Portal](https://kabe89.github.io/BioLabCalc/).

---

⭐ **Star BioLabCalc on GitHub**: [github.com/kabe89/BioLabCalc](https://github.com/kabe89/BioLabCalc)